## Processing Strava Data and GWR Features

In [57]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterstats as rs
import matplotlib.pyplot as plt
import os 
import rasterio as rio 
from rasterio.plot import show
from rasterio.merge import merge

## Count data preprocessing

In [3]:
strava_25_monthly = pd.read_csv("Data/Strava/strava_2025monthly_bankspeninsula_peds.csv")
strava_22_ped = pd.read_csv("Data/Strava/strava_2022_bankspeninsula_peds.csv")
strava_22_cycle = pd.read_csv("Data/Strava/strava_2022_bankspeninsula_cycling.csv")
strava_22_counts = pd.concat([strava_22_ped, strava_22_cycle], ignore_index=True,join='inner', axis=0)

In [4]:
#Collate to monthly bins
strava_22_counts['time'] = pd.to_datetime(strava_22_counts['hour'],format='mixed')
strava_22_counts['month'] = strava_22_counts['time'].dt.to_period('M')
strava_22_monthly = strava_22_counts.groupby(['month', 'edge_uid', 'activity_type', 'osm_reference_id']).agg({
    'total_trip_count': 'sum',
    'forward_trip_count': 'sum', 'reverse_trip_count': 'sum', 'forward_people_count': 'sum',
    'reverse_people_count': 'sum', 'forward_commute_trip_count': 'sum',
    'reverse_commute_trip_count': 'sum', 'forward_leisure_trip_count': 'sum',
    'reverse_leisure_trip_count': 'sum', 'forward_male_people_count': 'sum',
    'reverse_male_people_count': 'sum', 'forward_female_people_count': 'sum',
    'reverse_female_people_count': 'sum', 'forward_unspecified_people_count': 'sum',
    'reverse_unspecified_people_count': 'sum', 'forward_18_34_people_count': 'sum',
    'reverse_18_34_people_count': 'sum', 'forward_35_54_people_count': 'sum',
    'reverse_35_54_people_count': 'sum', 'forward_55_64_people_count': 'sum',
    'reverse_55_64_people_count': 'sum', 'forward_65_plus_people_count': 'sum',
    'reverse_65_plus_people_count': 'sum',
    'forward_average_speed_meters_per_second': 'mean',
    'reverse_average_speed_meters_per_second': 'mean',

    }).reset_index()


In [5]:
strava_monthly_counts = pd.concat([strava_22_monthly, strava_25_monthly], ignore_index=True,join='inner', axis=0)
def classify_strava_activity(activity_type):
    activity_text = "" if pd.isna(activity_type) else str(activity_type).lower()
    pedestrian_keywords = ["walk", "hike", "hiking", "trail", "run", "running", "trek", "parkrun"]
    cycle_keywords = ["cycle", "bike", "biking", "mtb", "mountain", "road", "bmx", "gravel"]

    if any(keyword in activity_text for keyword in cycle_keywords):
        return "cycle"
    if any(keyword in activity_text for keyword in pedestrian_keywords):
        return "pedestrian"
    return "other"

strava_monthly_counts["activity_class"] = strava_monthly_counts["activity_type"].apply(classify_strava_activity)
strava_monthly_counts.to_csv("Data/Strava/strava_2022_25_monthly_counts.csv", index=False)

## DEM Features

### DEM Tile Merge

In [42]:
data_dir = "Data/Supplementaries/kx-nz-8m-digital-elevation-model-2012-GTiff"

tile_names = ["PH.tif", "PI.tif", "QH.tif", "QI.tif"]
file_paths = [os.path.join(data_dir, tile) for tile in tile_names]

src_files_to_mosaic = []
for fp in file_paths:
    src = rio.open(fp)
    src_files_to_mosaic.append(src)

mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans
})

output_fp = os.path.join(data_dir, "merged_nz_dem.tif")
with rio.open(output_fp, "w", **out_meta) as dest:
    dest.write(mosaic)

for src in src_files_to_mosaic:
    src.close()

print(f"Successfully merged {len(tile_names)} tiles into: {output_fp}")


Successfully merged 4 tiles into: Data/Supplementaries/kx-nz-8m-digital-elevation-model-2012-GTiff\merged_nz_dem.tif


### Buffer Geometry

In [65]:
strava_id_map.head()

,edgeUID,osmId,source,geometry
0,38567784,440289756.0,ped,"LINESTRING (1552032.633 5144532.095, 1551983.0..."
1,38567820,440289756.0,ped,"LINESTRING (1560319.955 5146560.284, 1560226.6..."
2,38581111,26769833.0,ped,"LINESTRING (1566546.473 5158272.527, 1566508.7..."
3,38581172,27029338.0,ped,"LINESTRING (1565866.516 5158766.981, 1565691.1..."
4,38581250,817757847.0,ped,"LINESTRING (1566488.095 5160706.584, 1566518.9..."


In [69]:
strava_id_map = gpd.read_file("Data/Strava/strava_bikeped_osm.gpkg").to_crs(epsg=2193)
strava_gwr_features = strava_id_map[['edgeUID', 'osmId', 'geometry']]
strava_gwr_features['buff_geometry'] = strava_gwr_features.geometry.buffer(5)

### Zonal Statistics 

In [ ]:
with rio.open("Data/Supplementaries/kx-nz-8m-digital-elevation-model-2012-GTiff/merged_nz_dem.tif") as DEM_src:
    elevation_stats = rs.zonal_stats(
        strava_gwr_features['buff_geometry'], 
        DEM_src.read(1), 
        affine=DEM_src.transform, 
        stats=[ 
            'max',
            'range'])
strava_gwr_features['elevation_range'] = [stat['range'] for stat in elevation_stats]
strava_gwr_features['elevation_max'] = [stat['max'] for stat in elevation_stats]
strava_gwr_features['gradient'] = strava_gwr_features['elevation_range'] / strava_gwr_features.geometry.length


In [76]:
strava_gwr_features.head()

,edgeUID,osmId,geometry,buff_geometry,elevation_range,elevation_max,gradient
0,38567784,440289756.0,"LINESTRING (1552032.633 5144532.095, 1551983.0...","POLYGON ((1551984.311 5144514.358, 1551934.73 ...",NaN,NaN,NaN
1,38567820,440289756.0,"LINESTRING (1560319.955 5146560.284, 1560226.6...","POLYGON ((1560227.939 5146531.102, 1560134.668...",0.172156,5.727067,0.000174
2,38581111,26769833.0,"LINESTRING (1566546.473 5158272.527, 1566508.7...","POLYGON ((1566512.68 5158220.024, 1566512.342 ...",1.254961,11.063545,0.008436
3,38581172,27029338.0,"LINESTRING (1565866.516 5158766.981, 1565691.1...","POLYGON ((1565688.513 5158873.441, 1565688.495...",0.394020,12.222276,0.000908
4,38581250,817757847.0,"LINESTRING (1566488.095 5160706.584, 1566518.9...","POLYGON ((1566523.486 5160639.886, 1566556.577...",0.186630,20.177166,0.000468
